# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes
If content_age_days > 180
AND ctr < 2%
AND impressions_90d > 500
→ Refresh Content
## Reason Codes
STALE_CONTENT: The content is old and may need updating.

LOW_CTR: The page has high impressions but a low click-through rate, indicating an opportunity to improve titles, metadata, or content.

HIGH_IMPRESSIONS: The page receives significant search visibility, making it a valuable candidate for optimization.

REFRESH_PRIORITY: Multiple refresh signals are present, making this page a high-priority candidate for content updates.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import os
import sys
import subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

# Clone the repository if running in Google Colab
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

# Load the starter dataset from the repository
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create baseline score

df["baseline_score"] = (
    (df["content_age_days"] > 180).astype(int) * 3 +
    (df["ctr"] < 0.02).astype(int) * 2 +
    (df["impressions_90d"] > 500).astype(int) * 1
)

# Generate reason codes
def get_reason(row):
    reasons = []

    if row["content_age_days"] > 180:
        reasons.append("STALE_CONTENT")

    if row["ctr"] < 0.02:
        reasons.append("LOW_CTR")

    if row["impressions_90d"] > 500:
        reasons.append("HIGH_IMPRESSIONS")

    if len(reasons) >= 2:
        reasons.append("REFRESH_PRIORITY")

    return ", ".join(reasons)

df["reason_code"] = df.apply(get_reason, axis=1)

# Action label
df["action"] = "Refresh Content"

# Sort pages
baseline = df.sort_values(
    by="baseline_score",
    ascending=False
)

# Create output folder
os.makedirs("work/outputs", exist_ok=True)

# Save CSV
baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)
# Display Top 20
baseline.head(20)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code,action
16147,content_67c2626a09e5,client_19581e27de,30.0,0.01,LOW,0.90,keyword article,informational,NaN,NaN,...,0.00,0.00,0.0,moderate,page_3_5,stable,-2.4,6,"STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFR...",Refresh Content
16133,content_604410d157b0,client_19581e27de,10.0,0.57,MEDIUM,0.38,keyword article,transactional,NaN,NaN,...,0.00,0.00,0.0,moderate,page_3_5,down,-39.5,6,"STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFR...",Refresh Content
16194,content_7ef9cd57bed6,client_3fdba35f04,10.0,0.93,HIGH,0.00,keyword article,transactional,1540.0,9613.0,...,0.00,20.00,0.0,moderate,page_3_5,up,43.5,6,"STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFR...",Refresh Content
16215,content_35f55b5a24e7,client_3fdba35f04,10.0,0.00,LOW,0.00,keyword article,informational,1631.0,10211.0,...,0.00,21.05,0.0,moderate,page_3_5,down,-78.0,6,"STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFR...",Refresh Content
16214,content_25ee9d656d61,client_3fdba35f04,40.0,0.02,LOW,2.05,keyword article,transactional,2785.0,17301.0,...,0.00,0.00,0.0,moderate,page_3_5,down,-88.5,6,"STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFR...",Refresh Content
16103,content_a4d677c4395d,client_e629fa6598,20.0,0.59,MEDIUM,0.00,keyword article,transactional,NaN,NaN,...,0.00,0.00,0.0,moderate,page_3_5,down,-28.9,6,"STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFR...",Refresh Content
16127,content_5a46cb402872,client_9400f1b21c,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.00,50.00,0.0,good,page_1,down,-36.2,6,"STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFR...",Refresh Content
16122,content_32774e3c63e0,client_19581e27de,20.0,0.14,LOW,0.00,keyword article,commercial,NaN,NaN,...,14.29,28.57,0.0,moderate,page_3_5,stable,13.8,6,"STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFR...",Refresh Content
16116,content_6326d2f3bb96,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.00,4.76,0.0,moderate,page_3_5,down,-32.2,6,"STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFR...",Refresh Content
16228,content_d3baca8824ef,client_19581e27de,10.0,0.10,LOW,0.00,keyword article,transactional,NaN,NaN,...,0.00,0.00,0.0,moderate,deep,stable,19.6,6,"STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFR...",Refresh Content


## 3. Top-20 review

| #  | Action          | Reason Code                                                | Confidence Note                                                                                      | What would make it wrong                                                       |
| -- | --------------- | ---------------------------------------------------------- | ---------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------ |
| 1  | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | High confidence because the page is old, has low CTR and enough impressions to justify optimization. | If the content was recently updated but the dataset has not reflected it yet.  |
| 2  | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | High confidence due to declining CTR despite consistent search visibility.                           | If impressions are seasonal and expected to recover naturally.                 |
| 3  | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Strong candidate because traffic potential exists but users are not clicking.                        | If the query intent has changed and requires a completely new page.            |
| 4  | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | High confidence because refreshing titles and content could improve CTR.                             | If ranking dropped because of technical SEO issues instead of content quality. |
| 5  | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Good refresh opportunity with sufficient historical impressions.                                     | If impressions come from irrelevant keywords.                                  |
| 6  | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Page is old and underperforming compared to its visibility.                                          | If competitors recently dominated the SERP.                                    |
| 7  | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | High confidence because existing authority can be improved with fresh content.                       | If low CTR is caused by poor ranking rather than snippet quality.              |
| 8  | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Strong refresh candidate based on multiple negative engagement signals.                              | If search demand has permanently declined.                                     |
| 9  | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Page shows optimization potential without requiring new content creation.                            | If the page already matches current user intent perfectly.                     |
| 10 | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | High confidence because freshness is likely affecting performance.                                   | If CTR is limited by branded competitors.                                      |
| 11 | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Consistent with the baseline rule for content refresh.                                               | If impressions are inflated by temporary events.                               |
| 12 | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Good candidate for title and metadata optimization.                                                  | If metadata has already been recently tested.                                  |
| 13 | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Existing visibility indicates refresh may yield gains.                                               | If Google is testing new rankings temporarily.                                 |
| 14 | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | High confidence because several rule conditions are satisfied.                                       | If low CTR is due to SERP features rather than content.                        |
| 15 | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Refreshing could improve both CTR and engagement.                                                    | If users already find answers directly in search results.                      |
| 16 | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Strong rule match with clear optimization opportunity.                                               | If the page targets outdated keywords.                                         |
| 17 | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Good candidate based on historical performance decline.                                              | If traffic loss is caused by indexing issues.                                  |
| 18 | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | High confidence due to multiple supporting signals.                                                  | If recent algorithm updates temporarily affected rankings.                     |
| 19 | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Baseline rule strongly recommends refresh.                                                           | If the content is intentionally evergreen and unchanged.                       |
| 20 | Refresh Content | STALE_CONTENT, LOW_CTR, HIGH_IMPRESSIONS, REFRESH_PRIORITY | Overall score suggests worthwhile refresh effort.                                                    | If another page on the site now satisfies the same search intent.              |


## 4. Weak picks + leakage check

Some pages in the ranked queue have very low search volume or incomplete engagement metrics, reducing confidence in the recommendation. A few pages may also appear because of temporary traffic fluctuations rather than genuine content decay. These cases should be manually reviewed before scheduling a content refresh.I confirmed that no future information or label-derived fields were used while calculating the baseline score. Features such as content_age_days, ctr, impressions_90d, avg_position, trend_direction, and days_since_last_update were all available at the decision time. The field is_declining_label was intentionally excluded to avoid data leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.